In [8]:
import os
import json
import datetime
from google.colab import drive

drive.mount('/content/drive')

DIRECTORIO = '/content/drive/My Drive/LPII_2026'

def procesar_directorio_datos(directorio: str) -> dict:
    """
    Función modular estructurada que:
      - Explora el directorio configurado.
      - Filtra exclusivamente archivos .json o .csv.
      - Extrae metadatos (tamaño en KB y fecha de última modificación).
      - Parsea y valida el archivo 'inventario.json' si existe.
      - Retorna un diccionario estructurado listo para una API.
    """
    resultado_api = {
        "estado": "exitoso",
        "directorio": directorio,
        "timestamp_proceso": datetime.datetime.now().isoformat(),
        "archivos_procesados": [],
        "inventario": None
    }

    try:
        if not os.path.exists(directorio):
            raise FileNotFoundError(f"El directorio '{directorio}' no existe.")

        elementos = os.listdir(directorio)
        print(f"Explorando directorio: {directorio}\n")

        for nombre in elementos:
            ruta_completa = os.path.join(directorio, nombre)

            if not os.path.isfile(ruta_completa):
                continue

            _, extension = os.path.splitext(nombre)
            extension = extension.lower()

            if extension not in ['.json', '.csv']:
                continue

            tamano_bytes = os.path.getsize(ruta_completa)
            tamano_kb = round(tamano_bytes / 1024, 2)

            mtime_timestamp = os.path.getmtime(ruta_completa)
            fecha_modificacion = datetime.datetime.fromtimestamp(mtime_timestamp).strftime('%Y-%m-%d %H:%M:%S')

            info_archivo = {
                "nombre": nombre,
                "extension": extension,
                "tamano_kb": tamano_kb,
                "ultima_modificacion": fecha_modificacion
            }

            resultado_api["archivos_procesados"].append(info_archivo)
            print(f" └── [Archivo Válido] {nombre} ({tamano_kb} KB) - Modificado: {fecha_modificacion}")

            if nombre.lower() == "inventario.json":
                try:
                    with open(ruta_completa, 'r', encoding='utf-8') as f:
                        contenido_json = json.load(f)

                        if isinstance(contenido_json, list):
                            resultado_api["inventario"] = contenido_json
                            print(f" 'inventario.json' parseado correctamente (Registros: {len(contenido_json)})")
                        else:
                            print(f"  Advertencia: 'inventario.json' no contiene una lista válida en su raíz.")
                            resultado_api["inventario"] = {"error": "Formato inválido: se esperaba una lista"}
                except json.JSONDecodeError:
                    print(f" Error de sintaxis JSON en el archivo inventario.json.")
                    resultado_api["inventario"] = {"error": "JSON malformado"}

    except FileNotFoundError as e:
        print(f" Error: {e}")
        resultado_api["estado"] = "error"
        resultado_api["mensaje"] = str(e)
    except Exception as e:
        print(f" Error inesperado: {e}")
        resultado_api["estado"] = "error"
        resultado_api["mensaje"] = str(e)

    return resultado_api

datos_salida_api = procesar_directorio_datos(DIRECTORIO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Explorando directorio: /content/drive/My Drive/LPII_2026

 └── [Archivo Válido] Calculadora.csv (1.77 KB) - Modificado: 2026-10-27 16:07:47
 └── [Archivo Válido] CuentaRegresiva.json (1.29 KB) - Modificado: 2026-10-27 16:09:25
 └── [Archivo Válido] inventario (1).json (3.94 KB) - Modificado: 2026-10-27 16:24:28
 └── [Archivo Válido] inventario.json (0.28 KB) - Modificado: 2026-10-27 16:24:41
      'inventario.json' parseado correctamente (Registros: 3)
